# Assignment 1 — The Headline Test
## Question 2: Are recorded prices different between City 1 and City 2?

**Student:** Sulaim Niaz | **ERP:** 37389

**Claim:** “Prices are the same everywhere.”

**Research question:** Did the four selected products have different average recorded prices in City 1 and City 2 over the 12 observation dates?

**Variables:** `city`, `item`, `price`, `date`, and `unit`.

**Method:** Match the same product on the same date in both cities, compare average prices, calculate differences, and visualise the results.

**Data limitation:** The CSV labels itself as illustrative teaching data, not actual PBS observations. This notebook therefore describes only the supplied teaching dataset.

## 1. Import the libraries

Pandas reads and analyses tabular data. Matplotlib creates graphs.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

## 2. Load the CSV file

Keep `teaching_prices_clean.csv` in the same folder as this notebook. If it is somewhere else, replace the filename with the correct file path.

In [ ]:
df = pd.read_csv("teaching_prices_clean.csv")
df.head()

## 3. Inspect the dataset

Check its dimensions, column names, distinct values, missing values, duplicates and units.

In [ ]:
print("Shape (rows, columns):", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

print("\nDistinct products:", df["item"].nunique())
print("Products:", sorted(df["item"].unique()))

print("\nDistinct dates:", df["date"].nunique())
print("Distinct cities:", df["city"].nunique())

print("\nMissing values by column:")
print(df.isna().sum())

print("\nExact duplicate rows:", df.duplicated().sum())

print("\nUnits recorded:")
print(df["unit"].value_counts(dropna=False))

print("\nData labels:")
print(df["data_label"].value_counts(dropna=False))

## 4. Prepare the dates and check the comparison structure

Convert `date` to a date type. Then check that each product/date has one observation for each city. This is important because the comparison should match like with like.

In [ ]:
df["date"] = pd.to_datetime(df["date"])

# Count observations for each product/date/city combination
cell_counts = df.groupby(["date", "item", "city"]).size()
print("Minimum and maximum rows per date/item/city group:",
      cell_counts.min(), cell_counts.max())

print("Date range:", df["date"].min().date(), "to", df["date"].max().date())

## 5. Calculate average prices by product and city

The mean is calculated across the 12 observation dates for each product in each city.

In [ ]:
average_prices = (
    df.groupby(["item", "city"])["price"]
      .mean()
      .unstack("city")
)

average_prices = average_prices[["City 1", "City 2"]]
average_prices.round(2)

## 6. Compare matched observations on the same date

A fair comparison pairs the same product and date across City 1 and City 2. The difference is City 2 minus City 1. A positive result means the recorded price in City 2 is higher for that matched observation.

In [ ]:
matched = (
    df.pivot_table(
        index=["date", "item"],
        columns="city",
        values="price",
        aggfunc="mean"
    )
    .dropna(subset=["City 1", "City 2"])
    .reset_index()
)

matched["difference_City2_minus_City1"] = (
    matched["City 2"] - matched["City 1"]
)

matched["percentage_City2_higher"] = (
    matched["difference_City2_minus_City1"]
    / matched["City 1"] * 100
)

matched.head(10)

## 7. Summarise the difference for each product

The table reports each product's mean price in each city, the average absolute difference, and the average percentage difference across matched dates.

In [ ]:
product_comparison = (
    matched.groupby("item")
    .agg(
        City_1_mean=("City 1", "mean"),
        City_2_mean=("City 2", "mean"),
        Mean_difference=("difference_City2_minus_City1", "mean"),
        Mean_percentage_higher=("percentage_City2_higher", "mean")
    )
)

product_comparison.round(2)

## 8. Calculate the overall average difference

This is a descriptive comparison of all observations in the supplied dataset. It is not a national price index.

In [ ]:
city_average = df.groupby("city")["price"].mean()
overall_difference = city_average["City 2"] - city_average["City 1"]
overall_percentage = overall_difference / city_average["City 1"] * 100

print("City 1 overall average:", round(city_average["City 1"], 2))
print("City 2 overall average:", round(city_average["City 2"], 2))
print("Difference (City 2 - City 1):", round(overall_difference, 2))
print("Percentage above City 1:", round(overall_percentage, 2), "%")

## 9. Graph 1 — Average price by product and city

The grouped bar chart compares the average recorded price of each product in the two cities.

In [ ]:
average_prices.plot(
    kind="bar",
    figsize=(9, 5)
)

plt.title("Average Recorded Price by Product and City")
plt.xlabel("Product")
plt.ylabel("Average recorded price (PKR per kg)")
plt.xticks(rotation=0)
plt.legend(title="City")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## 10. Graph 2 — Matched price difference by product

This graph shows the average difference (City 2 minus City 1) for each product. Positive bars indicate higher recorded prices in City 2.

In [ ]:
mean_differences = matched.groupby("item")[
    "difference_City2_minus_City1"
].mean()

mean_differences.plot(
    kind="bar",
    figsize=(9, 5)
)

plt.axhline(0, linewidth=1)
plt.title("Average Matched Price Difference: City 2 minus City 1")
plt.xlabel("Product")
plt.ylabel("Average difference (PKR per kg)")
plt.xticks(rotation=0)
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## 11. Interpretation and conclusion

For the supplied CSV, the average price in City 1 is approximately PKR 199.17 and in City 2 approximately PKR 205.17. City 2's overall average is therefore PKR 6.00 higher, about 3.01% above City 1's average.

At product level, the mean prices are:
- Oil B: City 1 PKR 321.83; City 2 PKR 327.83.
- Pulse C: City 1 PKR 202.92; City 2 PKR 208.92.
- Staple A: City 1 PKR 108.17; City 2 PKR 114.17.
- Sweetener D: City 1 PKR 163.75; City 2 PKR 169.75.

City 2 is PKR 6 higher for every matched product/date pair in this teaching dataset. This is a descriptive pattern in the supplied file. It does not explain why the difference exists.

**Conclusion:** The claim “Prices are the same everywhere” is not supported by this dataset: the recorded average prices are higher in City 2 for all four products. However, the conclusion must be limited to these four products, two anonymised cities and 12 dates. The dataset is explicitly illustrative teaching data, not actual PBS observations, so it cannot establish real-world differences across Pakistan.

## 12. Limitations and questions the data cannot answer

- It cannot show that prices differ in every city or for every product.
- It cannot explain the cause of the price difference because factors such as transport costs, rent, supply and production costs are not included.
- It cannot show whether households can afford the products because income and expenditure data are absent.
- It is labelled illustrative teaching data, not actual PBS observations.
- Consult the course data dictionary for official definitions and any additional metadata.